In [66]:
import os
from google import genai
from google.genai import types
from dotenv import load_dotenv

load_dotenv()

True

In [16]:
client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

In [51]:
def llm(prompt):
    response = client.models.generate_content_stream(
        model = "gemini-2.5-flash-lite",
        contents = prompt
    )

    for chunk in response:
        print(chunk.text, end="", flush = True)

In [52]:
llm("Who won the Euro 2021?")

**Italy won the Euro 2020 (played in 2021) championship.**

In [53]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)

print(answer)

It's great you're interested in the course! To give you the best answer, I need a little more information. Please tell me:

*   **What is the name of the course you're referring to?**
*   **Where did you find out about it?** (e.g., a specific website, platform like Coursera or Udemy, a university, a company)
*   **Is there a specific start date mentioned for the course?**

Once I have these details, I can help you figure out if you can join now.None


In [54]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [55]:
rag_prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [56]:
answer_w_rag = llm(rag_prompt)
print(answer)

Yes, you can join now. If you wish to receive a certificate, ensure you submit your project before the submission deadline.None


# Course FAQ Dataset

In [23]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [24]:
courses_raw

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 404},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 85},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472}]

In [26]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1350

In [27]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

# Indexing

In [28]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [29]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [30]:
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'How should I start the course and follow the weekly workflow?',
 'When will the course be offered next?']

## Wrapping Function

In [31]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [36]:
question = "When can I start the course?"
results = search(question)

## Create Prompt

In [73]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [34]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [35]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")
    
    return "\n".join(lines).strip()

In [38]:
print(build_context(results))

General Course-Related Questions
Q: How should I start the course and follow the weekly workflow?
A: Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).

You can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).

A typical workflow is:

1. Watch the lesson videos.
2. Work through the lesson notebooks/code.
3. Read the homework instructions on GitHub.
4. Submit answers through the course platform before the deadline.

Homework is similar to the lesson flow, but uses a different dataset or slightly different task.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confi

In [39]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )

    return prompt.strip()

In [42]:
question = "When can I join?"
search_results = search(question)

prompt = build_prompt(question, search_results)

print(prompt)

Question:
When can I join?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

Module 1 Homework
Q: Where can I find the homework questions?
A: Homework links are available in the course GitHub repo and in the course management platform.

For the 2026 Module 1 homework, use:

- [Module 1 cohort materials](https://github.com/DataTalksClub/llm-zoomcamp/tree/main/cohorts/2026/01-agentic-rag)
- [Module 1 homework instructions](https://github.com

# Sending the Prompt

In [57]:
response = client.models.generate_content(
    model = "gemini-2.5-flash-lite",
    contents = "Who the top scorer in World Cup right now?"
)

In [62]:
# Calculating Token Price
input_price = 0.1 / 1_000_000
output_price = 0.40 / 1_000_000

input_token = response.usage_metadata.prompt_token_count
output_token = response.usage_metadata.candidates_token_count

In [63]:
print(f"Token Usage: ${input_price * input_token + output_token * output_price}")

Token Usage: $3.5900000000000005e-05


# LLM Function Updated

In practice, we would like to add the information such as INSTRUCTION and prompt in separate way.

In [70]:
def llm(instructions, user_prompt, model = "gemini-2.5-flash-lite"):
    config = types.GenerateContentConfig(
        system_instruction=instructions
    )

    response = client.models.generate_content(
        model=model,
        contents=user_prompt,
        config=config
    )

    return response.text

In [71]:
INSTRUCTIONS = """
You are association football pundit, you have all the knowledge about association football formation and strategy
"""

user_prompt = """
What is the best approach to score against a defensive team?
"""

llm(INSTRUCTIONS, user_prompt)

'Ah, a classic conundrum! Facing a well-drilled, defensively solid team is one of the toughest challenges in football, and it requires a multifaceted approach, not just a single "magic bullet." There\'s no one-size-fits-all answer, as the "best" approach will always be dictated by your own team\'s strengths, the specific characteristics of the opposition\'s defense, and the game situation. However, I can outline the key principles and strategic considerations that generally prove most effective.\n\nHere\'s my breakdown of the best approach to scoring against a defensive team:\n\n## 1. Patience and Possession: The Foundation\n\nThis is paramount. Defensive teams thrive on frustration. They want you to panic, to force passes, to commit too many players forward and leave yourselves exposed.\n\n*   **Dominating Possession:** The more the ball is at your feet, the less opportunity the opposition has to organize their defense, rest, or launch counter-attacks. Work the ball patiently from the

In [72]:
def rag(query, model="gemini-2.5-flash-lite"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model = model)
    return answer

In [74]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can join the course now. However, if you wish to receive a certificate, you must submit your project before the submission deadline.


In [75]:
answer2 = rag("How can I get the certificate?")
print(answer2)

You can get a certificate if you finish the course with a "live" cohort. You will not receive a certificate if you complete the course in a self-paced mode. To get a certificate, you need to peer-review 3 capstone projects after submitting your own project. This peer review can only be done while the course is running. Additionally, you need to pass the Capstone project.


In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from ingest import load_faq_data, build_index
from rag_helper import RAGBase
from google import genai

In [3]:
documents = load_faq_data()
index = build_index(documents)

In [6]:
gemini_client = genai.Client()
assistant = RAGBase(
    index=index,
    llm_client=gemini_client
)

In [7]:
answer = assistant.rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can join the course now. However, if you wish to receive a certificate, you must submit your project before the submission period closes.


In [8]:
assistant.rag("How do I get the certificate?")

'You can only get a certificate if you finish the course with a "live" cohort. Certificates are not awarded for self-paced modes. To receive a certificate, you must peer-review 3 capstone projects after submitting your own project. This peer review can only be done while the course is actively running. Additionally, you need to pass the Capstone project.'

In [5]:
from rag_helper import RAGBase
from google import genai
from dotenv import load_dotenv
load_dotenv()

gemini_client = genai.Client()

In [6]:
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

In [7]:
assistant = RAGBase(
    index=sqlite_index,
    llm_client=gemini_client
)

In [8]:
answer = assistant.rag("CAn I still join after it started?")
print(answer)

ANSWER: Iya, sampeyan isih iso gabung, nanging yen sampeyan pengin entuk sertifikat, sampeyan kudu ngirim proyek sampeyan nalika isih ditampa.


In [9]:
sqlite_index.close()